In [ ]:
# Cleaning the vehicle data (VEH0142)

# Load the vehicle data
# 1. Read the CSV from data/raw
# 2. Skip the 4 junk rows so row 5 becomes the headers
# 3. Treat commas in numbers as thousands separators (8,817 → 8817)
# 4. Treat [x] and [low] as missing values (NaN), not text
# 5. Read the whole file before guessing column types (low_memory=False)
# 6. Check the shape

In [1]:
import pandas as pd

In [2]:
vehicles = pd.read_csv("C:/Users/kraus/Desktop/Ironhack/ev-charging-gap/data/raw/veh0142.csv", skiprows=4, thousands=",", na_values="[x]", low_memory=False)

print(vehicles.shape)
vehicles.head()

(26156, 74)


,Units,BodyType,Fuel,Keepership,ONS Sort,ONS Code,ONS Geography,2026 Q2,2026 Q1,2025 Q4,...,2012 Q1,2011 Q4,2011 Q3,2011 Q2,2011 Q1,2010 Q4,2010 Q3,2010 Q2,2010 Q1,2009 Q4
0,Number,Buses and coaches,Battery electric,Company,1,K02000001,United Kingdom,"8,817","8,235","7,669",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Number,Buses and coaches,Battery electric,Company,2,K03000001,Great Britain,"8,549","7,988","7,425",...,54,58,57,59,58,49,45,41,40,38
2,Number,Buses and coaches,Battery electric,Company,3,E92000001,England,"7,333","6,804","6,276",...,51,55,54,57,55,46,43,39,38,36
3,Number,Buses and coaches,Battery electric,Company,4,E12000001,North East,211,170,142,...,7,12,12,13,14,7,5,4,4,4
4,Number,Buses and coaches,Battery electric,Company,5,E06000047,County Durham,1,1,1,...,[low],[low],3,3,3,3,1,[low],[low],[low]


In [ ]:
# Limitation: Very small values shown as [low] were treated as missing (about 15 cells in the analysed slice)

In [3]:
vehicles = pd.read_csv("../data/raw/veh0142.csv", skiprows=4, thousands=",", na_values=["[x]", "[low]"], low_memory=False)

print(vehicles.shape)
vehicles.head()

(26156, 74)


,Units,BodyType,Fuel,Keepership,ONS Sort,ONS Code,ONS Geography,2026 Q2,2026 Q1,2025 Q4,...,2012 Q1,2011 Q4,2011 Q3,2011 Q2,2011 Q1,2010 Q4,2010 Q3,2010 Q2,2010 Q1,2009 Q4
0,Number,Buses and coaches,Battery electric,Company,1,K02000001,United Kingdom,8817.0,8235.0,7669.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Number,Buses and coaches,Battery electric,Company,2,K03000001,Great Britain,8549.0,7988.0,7425.0,...,54.0,58.0,57.0,59.0,58.0,49.0,45.0,41.0,40.0,38.0
2,Number,Buses and coaches,Battery electric,Company,3,E92000001,England,7333.0,6804.0,6276.0,...,51.0,55.0,54.0,57.0,55.0,46.0,43.0,39.0,38.0,36.0
3,Number,Buses and coaches,Battery electric,Company,4,E12000001,North East,211.0,170.0,142.0,...,7.0,12.0,12.0,13.0,14.0,7.0,5.0,4.0,4.0,4.0
4,Number,Buses and coaches,Battery electric,Company,5,E06000047,County Durham,1.0,1.0,1.0,...,NaN,NaN,3.0,3.0,3.0,3.0,1.0,NaN,NaN,NaN


In [ ]:
# Keep only private battery electric cars
# 1. Check each row: is BodyType "Cars"?
# 2. Check each row: is Fuel "Battery electric"?
# 3. Check each row: is Keepership "Private"?
# 4. Keep only the rows where all three are True (& means "and")
# 5. Check the shape

In [4]:
cars = vehicles[
    (vehicles["BodyType"] == "Cars") &
    (vehicles["Fuel"] == "Battery electric") &
    (vehicles["Keepership"] == "Private")
]

print(cars.shape)

(447, 74)


In [5]:
# Keep only the lowest geography level (individual councils)
# 1. List the code prefixes for councils in England, Scotland, Wales and NI
# 2. Take the first 3 characters of each area code
# 3. Keep rows whose prefix is in the list
#    (this also drops the [z] "unknown location" rows)
# 4. Check the shape

lowest_tier = ["E06", "E07", "E08", "E09", "S12", "W06", "N09"]

cars_la = cars[cars["ONS Code"].str[:3].isin(lowest_tier)]

print(cars_la.shape)

(361, 74)


In [6]:
# Sanity check: do councils + unknown-location cars add up to the UK total?
# 1. Add up cars across all councils for the latest quarter (2026 Q2)
# 2. Add up the [z] "unknown location" rows for the same quarter
# 3. Look up the published UK total
# 4. Councils + unknown should equal the UK total exactly

print(cars_la["2026 Q2"].sum())
print(cars[cars["ONS Code"] == "[z]"]["2026 Q2"].sum())
print(cars[cars["ONS Code"] == "K02000001"]["2026 Q2"])

954495.0
5514.0
1710    960009.0
Name: 2026 Q2, dtype: float64


In [7]:
# Tidy up the columns
# 1. Drop the columns we no longer need: the slices (now all the same),
#    Units (always "Number") and ONS Sort (DfT's row order)
# 2. Rename the label columns to match the charger tables
# 3. Strip the indent spaces from council names (" County Durham" → "County Durham")
# 4. Check: expect (361, 69) and names without leading spaces

cars_la = cars_la.drop(columns=["Units", "BodyType", "Fuel", "Keepership", "ONS Sort"])

cars_la = cars_la.rename(columns={
    "ONS Code": "area_code",
    "ONS Geography": "area_name"
})

cars_la["area_name"] = cars_la["area_name"].str.strip()

print(cars_la.shape)
cars_la.head()

(361, 69)


,area_code,area_name,2026 Q2,2026 Q1,2025 Q4,2025 Q3,2025 Q2,2025 Q1,2024 Q4,2024 Q3,...,2012 Q1,2011 Q4,2011 Q3,2011 Q2,2011 Q1,2010 Q4,2010 Q3,2010 Q2,2010 Q1,2009 Q4
1714,E06000047,County Durham,7662.0,6712.0,5905.0,5414.0,4780.0,4252.0,3769.0,3433.0,...,4.0,4.0,4.0,3.0,3.0,1.0,1.0,1.0,1.0,1.0
1715,E06000005,Darlington,1507.0,1317.0,1162.0,1068.0,924.0,842.0,722.0,663.0,...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1716,E06000001,Hartlepool,1029.0,882.0,763.0,685.0,594.0,523.0,448.0,420.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1717,E06000002,Middlesbrough,1207.0,1083.0,942.0,847.0,731.0,628.0,528.0,471.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1718,E06000057,Northumberland,5974.0,5304.0,4737.0,4355.0,3919.0,3529.0,3177.0,2973.0,...,6.0,6.0,7.0,6.0,1.0,NaN,NaN,NaN,NaN,NaN


In [8]:
# Reshape from wide to long
# 1. Keep area_code and area_name on every row
# 2. Turn the 67 quarter columns into rows (one row per council per quarter)
# 3. The quarter labels ("2026 Q2") already match the charger tables,
#    so name that column join_quarter straight away
# 4. Name the number column bevs (battery electric vehicles)
# 5. Check the shape: expect (24187, 4), because 361 councils × 67 quarters

cars_long = cars_la.melt(
    id_vars=["area_code", "area_name"],
    var_name="join_quarter",
    value_name="bevs"
)

print(cars_long.shape)
cars_long.head()

(24187, 4)


,area_code,area_name,join_quarter,bevs
0,E06000047,County Durham,2026 Q2,7662.0
1,E06000005,Darlington,2026 Q2,1507.0
2,E06000001,Hartlepool,2026 Q2,1029.0
3,E06000002,Middlesbrough,2026 Q2,1207.0
4,E06000057,Northumberland,2026 Q2,5974.0


In [9]:
# Save the clean table
# Save to data/clean as private_bevs.csv, without row numbers

cars_long.to_csv("../data/clean/private_bevs.csv", index=False)

In [10]:
# Test run before the join: do the two tables share the same councils?
# 1. Load the clean charger table
# 2. Make a set of area codes from each table (a set = a list with no duplicates)
# 3. Find the codes that appear in both (& = "in both")
# 4. Expect 361: every council in the vehicle table also appears in the charger table

chargers = pd.read_csv("../data/clean/chargers_all_speeds.csv")

shared_codes = set(cars_long["area_code"]) & set(chargers["area_code"])

print(len(shared_codes))

361
